#Task 2: Feb Internship NLP- Sentiment Analysis using NLP Pipeline & ML Models

In [2]:
# import required libraries
import pandas as pd
from google.colab import files
# load dataset
uploaded = files.upload()
df = pd.read_csv('IMDB Dataset.csv')
df.head()

# check dataset info
print("Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

Saving IMDB Dataset.csv to IMDB Dataset (1).csv
Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


##NLP Preprocessing

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# function to clean and preprocess text data
def preprocess_text(text):

    # convert to lowercase
    text = text.lower()

    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove punctuation & special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # tokenization
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

# apply preprocessing
df['clean_text'] = df['review'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


##Feature Engineering

In [7]:
# converting text into numerical features using BoW and TF-IDF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Bag of Words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

# target
y = df['sentiment']

##Train-Test Split

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

##Model Building

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

DecisionTreeClassifier()

##Evaluation

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# function to evaluate model performance using metrics
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("-"*40)

print("Logistic Regression:")
evaluate(lr, X_test, y_test)

print("Naive Bayes:")
evaluate(nb, X_test, y_test)

print("Decision Tree:")
evaluate(dt, X_test, y_test)

Logistic Regression:
Accuracy: 0.8841
Precision: 0.8844234802655033
Recall: 0.8841
F1 Score: 0.8840599019111612
----------------------------------------
Naive Bayes:
Accuracy: 0.8458
Precision: 0.8458299170400514
Recall: 0.8458
F1 Score: 0.8457884267539079
----------------------------------------
Decision Tree:
Accuracy: 0.7119
Precision: 0.7119349978943986
Recall: 0.7119
F1 Score: 0.7119043359581155
----------------------------------------


##Comparison Table

In [11]:
results = []

def get_metrics(model, name):
    y_pred = model.predict(X_test)
    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average='weighted'),
        recall_score(y_test, y_pred, average='weighted'),
        f1_score(y_test, y_pred, average='weighted')
    ])

get_metrics(lr, "Logistic Regression")
get_metrics(nb, "Naive Bayes")
get_metrics(dt, "Decision Tree")

import pandas as pd
df_results = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score"
])

df_results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8841,0.884423,0.8841,0.884060
1,Naive Bayes,0.8458,0.845830,0.8458,0.845788
2,Decision Tree,0.7119,0.711935,0.7119,0.711904


## Insights

- Logistic Regression performed the best because it handles high-dimensional sparse data effectively.
- TF-IDF provided better results than Bag of Words by reducing the impact of common words.
- Naive Bayes performed well and is computationally efficient but slightly less accurate.
- Decision Tree showed lower performance due to overfitting on text data.

This project demonstrates a complete NLP pipeline for sentiment analysis.
Among all models, Logistic Regression performed best due to better generalization on text data.
The pipeline effectively transforms raw text into meaningful features for ML models.